In [14]:
# === Cell 1: imports & path ===
import os
import sys
try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from power_traces_anova_f_traces_vis import (
    plot_anova_F_traces_for_roi,
    plot_per_electrode_F_traces,
    plot_per_electrode_power_traces,
)
from src.analysis.power.power_traces import load_significant_electrodes
from src.analysis.config.condition_registry import get_conditions_obj

from pathlib import Path
from src.analysis.utils.general_utils import create_subjects_mne_objects_dict

# Adjust to your data root
DERIV = os.path.join(project_root, 'dcc_scripts', 'power', 'figs')
# EPOCHS_ROOT = 'Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit'
EPOCHS_ROOT = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit"
# EPOCHS_ROOT = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_13.0-30.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

CONDITION_LABELS = ['stimulus_lwpc_conditions', 'stimulus_lwps_conditions', 'stimulus_congruency_by_switch_proportion_conditions', 'stimulus_switch_type_by_incongruent_proportion_conditions']
ROI = 'lpfc'

# If your run nests anova_F_traces under anova_within_roi/, the parent is that
# subdir; otherwise just use RUN_DIR (= DERIV / EPOCHS_ROOT).
RUN_DIR              = os.path.join(DERIV, EPOCHS_ROOT)
ANOVA_WITHIN_ROI_DIR = os.path.join(RUN_DIR, 'anova_within_roi')
F_TRACES_DIR         = os.path.join(ANOVA_WITHIN_ROI_DIR, 'anova_F_traces')
evk_dir              = Path(ANOVA_WITHIN_ROI_DIR) / ROI
WINDOW_SIZE, STEP_SIZE = 64, 16

# Pre-build the conditions dict per condition_label so cells below can use it.
conditions_by_label = {cl: get_conditions_obj(cl) for cl in CONDITION_LABELS}

# === Cell 0.5: project-level config (run once near the top) ===
SUBJECTS = ['D0057', 'D0059', 'D0063', 'D0065', 'D0069', 'D0071', 'D0077', 'D0090',
            'D0094', 'D0100', 'D0102', 'D0103', 'D0107A', 'D0110', 'D0116', 'D0117',
            'D0121', 'D0133', 'D0134']
TASK            = 'GlobalLocal'
LAB_ROOT        = None        # auto-detect
ACC_TRIALS_ONLY = True
MNE_OBJ_TYPE    = 'HG_ev1_power_rescaled'

conditions_by_label = {cl: get_conditions_obj(cl) for cl in CONDITION_LABELS}

In [15]:

for CONDITION_LABEL in CONDITION_LABELS:
    CONDITIONS_SAVE_NAME = f'{CONDITION_LABEL}_19_subjects'

    # === Cell 2: load metadata (optional) ===
    meta_path = os.path.join(ANOVA_WITHIN_ROI_DIR, f'{CONDITION_LABEL}_19_subjects_metadata.json')

    meta = json.load(open(meta_path))
    print('factors:', meta.get('anova_factors'))
    print('rois:   ', meta.get('rois'))
    print('sfreq:  ', meta.get('sfreq'))

    evk_files = sorted(evk_dir.glob(f'{CONDITIONS_SAVE_NAME}_*_{ROI}_evoked.npz'))
    if not evk_files:
        raise FileNotFoundError(f'no evoked .npz in {evk_dir}')
    times = np.load(evk_files[0])['times']

    n_windows = (len(times) - WINDOW_SIZE) // STEP_SIZE + 1
    window_centers = np.array([times[i * STEP_SIZE + WINDOW_SIZE // 2]
                            for i in range(n_windows)])

    fig = plot_anova_F_traces_for_roi(
        save_dir=ANOVA_WITHIN_ROI_DIR,                                       # NOT F_TRACES_DIR
        roi=ROI,
        conditions_save_name=CONDITIONS_SAVE_NAME,
        window_centers=window_centers,
        save_path=os.path.join(F_TRACES_DIR, f'{CONDITIONS_SAVE_NAME}_f_traces.png'),
    )

factors: ['congruency', 'incongruentProportion']
rois:    ['lpfc', 'occ']
sfreq:   256
No F-traces found for lpfc.
factors: ['switchType', 'switchProportion']
rois:    ['lpfc', 'occ']
sfreq:   256
No F-traces found for lpfc.
factors: ['congruency', 'switchProportion']
rois:    ['lpfc', 'occ']
sfreq:   256
No F-traces found for lpfc.
factors: ['switchType', 'incongruentProportion']
rois:    ['lpfc', 'occ']
sfreq:   256
No F-traces found for lpfc.


In [16]:
# === Cell 4: within-electrode F-trace grids ===
from pathlib import Path
from src.analysis.vis.power_traces_anova_f_traces_vis import (
    plot_per_electrode_F_traces, plot_per_electrode_power_traces,
)
from src.analysis.power.power_traces import load_significant_electrodes

WITHIN_ROI = 'lpfc'     # only 'occ' and 'lpfc' exist on disk for this band
# EFFECT_BY_CONDITION = {
#     'stimulus_lwpc_conditions':                              'C(congruency)',
#     'stimulus_lwps_conditions':                              'C(switchType)',
#     'stimulus_congruency_by_switch_proportion_conditions':   'C(congruency)',
#     'stimulus_switch_type_by_incongruent_proportion_conditions': 'C(switchType)',
# }

WITHIN_ELEC_ROOT = Path(RUN_DIR) / 'anova_within_electrode' / 'within_elec_anova'
PER_ELEC_FIG_DIR = Path(ANOVA_WITHIN_ROI_DIR) / 'per_electrode_F_traces' / WITHIN_ROI
PER_ELEC_FIG_DIR.mkdir(parents=True, exist_ok=True)

# === Cell 4: within-electrode F-trace grids — all effects ===
for CONDITION_LABEL in CONDITION_LABELS:
    CONDITIONS_SAVE_NAME = f'{CONDITION_LABEL}_19_subjects'
    within_run = WITHIN_ELEC_ROOT / CONDITIONS_SAVE_NAME

    # Discover effects from the first available npz.
    npzs = sorted((within_run / WITHIN_ROI).glob('*/*.npz'))
    if not npzs:
        print(f'{CONDITION_LABEL}: no electrode npzs, skipping')
        continue
    effect_names = list(np.load(npzs[0])['effect_names'])
    print(f'{CONDITION_LABEL}: effects = {effect_names}')

    for effect in effect_names:
        safe_eff = effect.replace(':', '_x_').replace('C(', '').replace(')', '')
        cond_save_dir = PER_ELEC_FIG_DIR / f'{CONDITIONS_SAVE_NAME}_{safe_eff}'
        cond_save_dir.mkdir(parents=True, exist_ok=True)

        pages = plot_per_electrode_F_traces(
            within_run_dir=within_run,
            roi=WITHIN_ROI,
            effect=effect,
            channels_per_page=20,
            save_dir=cond_save_dir,
            sample_window_centers=window_centers,
        )
        print(f'  {effect}: {len(pages)} page(s) -> {cond_save_dir}')


stimulus_lwpc_conditions: effects = [np.str_('C(congruency)'), np.str_('C(incongruentProportion)'), np.str_('C(congruency):C(incongruentProportion)')]
  C(congruency): 10 page(s) -> /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/power/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit/anova_within_roi/per_electrode_F_traces/lpfc/stimulus_lwpc_conditions_19_subjects_congruency
  C(incongruentProportion): 10 page(s) -> /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/power/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit/anova_within_roi/per_electrode_F_traces/lpfc/stimulus_lwpc_conditions_19_subjects_incongruentProportion
  C(congruency):C(incongruentProportion): 10 page(s) -> /hpc/group/coganlab/jz421/GlobalL

In [17]:
# === Cell 5: sig electrodes for every effect ===
sig_by_condition_effect = {}
for CONDITION_LABEL in CONDITION_LABELS:
    CONDITIONS_SAVE_NAME = f'{CONDITION_LABEL}_19_subjects'
    within_run = WITHIN_ELEC_ROOT / CONDITIONS_SAVE_NAME
    npzs = sorted((within_run / WITHIN_ROI).glob('*/*.npz'))
    if not npzs:
        continue
    effect_names = list(np.load(npzs[0])['effect_names'])

    has_summary = (within_run / 'summary.csv').exists() \
                  or (within_run / 'significant_effects_structure.json').exists()
    for effect in effect_names:
        if has_summary:
            sig = load_significant_electrodes(within_run, roi=WITHIN_ROI, effect=effect)
            tag = 'FDR'
        else:
            sig = _sig_from_npzs(within_run, WITHIN_ROI, effect)
            tag = 'uncorrected (npz scan)'
        sig_by_condition_effect[(CONDITION_LABEL, effect)] = sig
        print(f'{CONDITION_LABEL} / {effect}: {len(sig)} sig in {WITHIN_ROI}  [{tag}]')

stimulus_lwpc_conditions / C(congruency): 14 sig in lpfc  [FDR]
stimulus_lwpc_conditions / C(incongruentProportion): 29 sig in lpfc  [FDR]
stimulus_lwpc_conditions / C(congruency):C(incongruentProportion): 10 sig in lpfc  [FDR]
stimulus_lwps_conditions / C(switchType): 24 sig in lpfc  [FDR]
stimulus_lwps_conditions / C(switchProportion): 15 sig in lpfc  [FDR]
stimulus_lwps_conditions / C(switchType):C(switchProportion): 5 sig in lpfc  [FDR]
stimulus_congruency_by_switch_proportion_conditions / C(congruency): 19 sig in lpfc  [FDR]
stimulus_congruency_by_switch_proportion_conditions / C(switchProportion): 37 sig in lpfc  [FDR]
stimulus_congruency_by_switch_proportion_conditions / C(congruency):C(switchProportion): 9 sig in lpfc  [FDR]
stimulus_switch_type_by_incongruent_proportion_conditions / C(switchType): 37 sig in lpfc  [FDR]
stimulus_switch_type_by_incongruent_proportion_conditions / C(incongruentProportion): 32 sig in lpfc  [FDR]
stimulus_switch_type_by_incongruent_proportion_condi

In [18]:
# === Cell 6: per-electrode power-trace grids ===
from pathlib import Path
import gc
import numpy as np
from src.analysis.config.plotting_parameters import plotting_parameters

POWER_FIG_DIR = Path(ANOVA_WITHIN_ROI_DIR) / 'per_electrode_power_traces' / WITHIN_ROI
POWER_FIG_DIR.mkdir(parents=True, exist_ok=True)

def _safe_eff(eff):
    return eff.replace(':', '_x_').replace('C(', '').replace(')', '')

for CONDITION_LABEL in CONDITION_LABELS:
    CONDITIONS_SAVE_NAME = f'{CONDITION_LABEL}_19_subjects'
    within_run = WITHIN_ELEC_ROOT / CONDITIONS_SAVE_NAME

    # Every electrode that had ANOVA computed for this condition.
    all_pairs = [(p.parent.name, p.stem)
                 for p in sorted((within_run / WITHIN_ROI).glob('*/*.npz'))]
    if not all_pairs:
        print(f'{CONDITION_LABEL}: no electrode npzs, skipping')
        continue

    # Effects available for this condition (from the first npz).
    effect_names = list(np.load(within_run / WITHIN_ROI / all_pairs[0][0] / f'{all_pairs[0][1]}.npz')['effect_names'])

    # Sig pairs per effect (gathered in Cell 5).
    sig_pairs_by_effect = {
        eff: sig_by_condition_effect.get((CONDITION_LABEL, eff), [])
        for eff in effect_names
    }

    # Union of subjects we need to load for this condition: any subject appearing
    # in all_pairs OR in any of the sig sets.
    subs_needed = sorted({sub for sub, _ in all_pairs}
                         | {sub for pairs in sig_pairs_by_effect.values() for sub, _ in pairs})

    conditions      = conditions_by_label[CONDITION_LABEL]
    condition_names = list(conditions.keys())

    print(f'\n=== {CONDITION_LABEL}: loading MNE objects for {len(subs_needed)} subjects ===')
    raw_mne = create_subjects_mne_objects_dict(
        subjects=subs_needed,
        epochs_root_file=EPOCHS_ROOT,
        conditions=conditions,
        task=TASK,
        just_HG_ev1_rescaled=True,
        LAB_root=LAB_ROOT,
        acc_trials_only=ACC_TRIALS_ONLY,
    )

    # Flatten to [sub][cname] -> Epochs (or None) as the plotter expects.
    subjects_mne_objects = {
        sub: {cname: raw_mne.get(sub, {}).get(cname, {}).get(MNE_OBJ_TYPE)
              for cname in condition_names}
        for sub in subs_needed
    }

    # --- all electrodes (effect-independent) ---
    all_elecs_for_roi = {WITHIN_ROI: {}}
    for sub, elec in all_pairs:
        all_elecs_for_roi[WITHIN_ROI].setdefault(sub, []).append(elec)

    all_save_dir = POWER_FIG_DIR / CONDITIONS_SAVE_NAME / 'all_elecs'
    all_save_dir.mkdir(parents=True, exist_ok=True)
    pages = plot_per_electrode_power_traces(
        subjects_mne_objects=subjects_mne_objects,
        rois=[WITHIN_ROI],
        condition_names=condition_names,
        electrodes_per_subject_roi=all_elecs_for_roi,
        plotting_parameters=plotting_parameters,
        channels_per_page=20,
        save_dir=all_save_dir,
    )
    print(f'  all_elecs: {len(pages.get(WITHIN_ROI, []))} page(s) -> {all_save_dir}')

    # --- sig electrodes, one grid per effect ---
    for eff in effect_names:
        sig_pairs = sig_pairs_by_effect[eff]
        if not sig_pairs:
            print(f'  sig_elecs / {eff}: 0 electrodes, skipping')
            continue

        sig_elecs_for_roi = {WITHIN_ROI: {}}
        for sub, elec in sig_pairs:
            sig_elecs_for_roi[WITHIN_ROI].setdefault(sub, []).append(elec)

        sig_save_dir = POWER_FIG_DIR / CONDITIONS_SAVE_NAME / f'sig_elecs_{_safe_eff(eff)}'
        sig_save_dir.mkdir(parents=True, exist_ok=True)
        pages = plot_per_electrode_power_traces(
            subjects_mne_objects=subjects_mne_objects,
            rois=[WITHIN_ROI],
            condition_names=condition_names,
            electrodes_per_subject_roi=sig_elecs_for_roi,
            plotting_parameters=plotting_parameters,
            channels_per_page=20,
            save_dir=sig_save_dir,
        )
        print(f'  sig_elecs / {eff}: {len(pages.get(WITHIN_ROI, []))} page(s) -> {sig_save_dir}')

    del raw_mne, subjects_mne_objects
    gc.collect()


=== stimulus_lwpc_conditions: loading MNE objects for 16 subjects ===
Loading data for subject: D0057


Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0057/D0057_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation matrices available
Adding metadata with 29 columns
449 matching events found
No baseline correction applied
0 projection items activated
Replacing existing metadata with 29 columns
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0057/D0057_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_power_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation matrices available


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c25: 143 valid trials out of 143
  Loading condition: Stimulus_c75
Adding metadata with 30 columns
55 matching events found
No baseline correction applied
    Stimulus_c75: 55 valid trials out of 55


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


...

    Stimulus_s_in_75incongruentBlock: 102 valid trials out of 102
  Loading condition: Stimulus_r_in_25incongruentBlock
Adding metadata with 30 columns
109 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_r_in_25incongruentBlock: 109 valid trials out of 109
  Loading condition: Stimulus_r_in_75incongruentBlock
Adding metadata with 30 columns
108 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_r_in_75incongruentBlock: 108 valid trials out of 108
  all_elecs: 10 page(s) -> /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/power/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit/anova_within_roi/per_electrode_power_traces/lpfc/stimulus_switch_type_by_incongruent_proportion_conditions_19_subjects/all_elecs
  sig_elecs / C(switchType): 2 page(s) -> /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/power/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_4.0-8.0_Hz_padLength_1.5s_bandpass_stat_func_ttest_ind_equal_var_False_nan_policy_omit/anova_within_roi/per_electrode_power_traces/lpfc/stimulus_switch_type_by_incongruent_proportion_conditions_19_subjects/sig_elecs_switchType
  sig_elecs / C(incongruentProportion): 2 page(s) -> /hpc/group/coganlab/jz421/GlobalLocal/dcc_scripts/pow